## llama的RoPE实现

In [6]:
import torch

def precompute_freqs_cis(
    head_dim: int,
    seq_len: int,
    theta: float = 10000.0,
    device=None,
):
    """
    预计算 LLaMA RoPE 的复数旋转因子
    返回形状: [seq_len, head_dim // 2] (complex64/complex128)
    """
    assert head_dim % 2 == 0, "head_dim 必须是偶数"
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device)[:(head_dim//2)].float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    angles = torch.outer(t, freqs)  # [seq_len, head_dim//2]
    freqs_cis = torch.polar(torch.ones_like(angles), angles)  # e^{i * angle}
    return freqs_cis


def _reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    """
    x: [bs, seq_len, n_heads, head_dim//2] (complex)
    freqs_cis: [seq_len, head_dim//2] (complex)
    """
    return freqs_cis.unsqueeze(0).unsqueeze(2)  # [1, seq_len, 1, head_dim//2]


def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
):
    """
    xq, xk: [bs, seq_len, n_heads, head_dim]
    freqs_cis: [seq_len, head_dim//2]
    返回: 旋转后 xq, xk，形状不变
    """
    assert xq.shape == xk.shape
    bs, seq_len, n_heads, head_dim = xq.shape
    assert head_dim % 2 == 0, "head_dim 必须是偶数"

    # 转为复数表示: [..., head_dim//2, 2] -> complex
    xq_c = torch.view_as_complex(xq.float().reshape(bs, seq_len, n_heads, head_dim // 2, 2))
    xk_c = torch.view_as_complex(xk.float().reshape(bs, seq_len, n_heads, head_dim // 2, 2))

    freqs = _reshape_for_broadcast(freqs_cis[:seq_len].to(xq_c.device), xq_c)

    xq_out = torch.view_as_real(xq_c * freqs).flatten(-2)
    xk_out = torch.view_as_real(xk_c * freqs).flatten(-2)

    return xq_out.type_as(xq), xk_out.type_as(xk)

In [7]:
dim = 17
a = torch.arange(0, dim, 2)
print(a)
print(a[:(dim//2)])
print(dim//2)

tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16])
tensor([ 0,  2,  4,  6,  8, 10, 12, 14])
8
